# 02 · GPU timing run (Whole model, EfficientNet-B0)

Assignment 1, step 2. Before spending the GPU budget on a full training run, measure how fast the author's
Whole model trains on a Kaggle GPU.

- Code: my fork of the author's training repo at a fixed commit (resize fix, autocast fix, timm pin).
- Data: the author's public datasets, attached as inputs (rectified fold 0 and 1, sparse masks), plus the
  competition files for ground truth.
- Sample: one epoch on 400 training images (fold 1) and 100 validation images (fold 0), with the author's
  settings (batch 4, AdamW 1e-3, BCE pos_weight 20, fp16 autocast, gradient checkpointing).
- Output: seconds per image for training and validation, and the estimated time of a full epoch
  (7,028 train / 1,764 val images).

In [ ]:
REPO = "https://github.com/darren1sheehan/physionet-image-digitization-Assignment-1.git"
COMMIT = "0c62252"
N_TRAIN, N_VAL = 400, 100              # sampled images for the timing epoch
FULL_TRAIN, FULL_VAL = 7028, 1764      # folds 1-4 train, fold 0 val

!git clone -q $REPO /kaggle/working/repo && cd /kaggle/working/repo && git checkout -q $COMMIT && git log --oneline -1
!pip install -q "timm<1.0.29" "segmentation-models-pytorch==0.5.0" "albumentations==2.0.8" omegaconf
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os; print("CPUs:", os.cpu_count())

In [ ]:
# Locate the attached inputs (Kaggle mount paths have changed over time, so search instead of hard-coding)
import glob, os
from pathlib import Path

def find_dir(name):
    for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):
        hits = [p for p in glob.glob(f"/kaggle/input/{depth}/{name}") if os.path.isdir(p)]
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(name)

rect = {f: find_dir(f"rectified_fold{f}") for f in (0, 1)}
mask_dir = find_dir("mask_x5000_500hz_comp_synthesis")
comp_train = next(p for p in (find_dir("train"),) if any(p.glob("*/*.csv")))
print(rect, mask_dir, comp_train, sep="\n")

In [ ]:
# Fold CSV restricted to a sample of fold 0 (val) and fold 1 (train), with paths pointing at the inputs
import pandas as pd

os.chdir("/kaggle/working/repo")
df = pd.read_csv("train_fold_with_synthesis.csv", dtype={"id": str, "type_id": str})
comp = df[~df["is_synthesis"]].copy()
comp["image_path"] = [str(rect.get(f, Path("missing")) / Path(p).name) for f, p in zip(comp["fold"], comp["image_path"])]
sample = pd.concat([
    comp[comp["fold"] == 1].sample(N_TRAIN, random_state=0),
    comp[comp["fold"] == 0].sample(N_VAL, random_state=0),
])
assert sample["image_path"].map(os.path.exists).all(), "some images are missing"
sample.to_csv("/kaggle/working/timing_folds.csv", index=False)

# Ground truth: competition train/{id}/{id}.csv -> ecg_dir/{id}.csv
ecg_dir = Path("/kaggle/working/ecg_csv"); ecg_dir.mkdir(exist_ok=True)
for i in sample.loc[sample["fold"] == 0, "id"].unique():
    dst = ecg_dir / f"{i}.csv"
    if not dst.exists():
        dst.symlink_to(comp_train / i / f"{i}.csv")
print(sample["fold"].value_counts().to_dict())

In [ ]:
# Data loading alone: if this is close to the training step time, the CPUs (not the GPU) are the bottleneck
import time, sys
sys.path.insert(0, "/kaggle/working/repo")
from src.data.dataset_whole import create_whole_dataloader

loader = create_whole_dataloader(df=sample[sample["fold"] == 1], df_synthesis=sample.iloc[:0], mask_dir=mask_dir,
                                 batch_size=4, offset=416, is_train=True, num_workers=4)
t, n = None, 0
for b, batch in enumerate(loader):
    if b == 2:
        t = time.time()                # skip worker start-up
    elif b > 2:
        n += 4
    if b == 32:
        break
load_s_per_img = (time.time() - t) / n
print(f"data loading only: {load_s_per_img:.2f} s/image")

In [ ]:
# One training epoch + validation with the author's settings, on the sample
import re, subprocess

cmd = ["python", "train.py", "--config", "configs/whole_model.yaml", "--output_dir", "/kaggle/working/outputs/timing",
       "data.csv_path=/kaggle/working/timing_folds.csv", f"data.mask_dir={mask_dir}", f"data.ecg_dir={ecg_dir}",
       "wandb.enabled=false", "training.num_epochs=1", "num_workers=4"]
t0 = time.time()
proc = subprocess.run(cmd, capture_output=True, text=True)
wall = time.time() - t0
log = proc.stdout + proc.stderr
print(log[-6000:])
assert proc.returncode == 0, "training failed"

In [ ]:
# Timing summary and full-scale estimate
def phase_seconds(name):
    # last tqdm line for the phase, e.g. "Epoch 0 [Train]: 100%|...| 100/100 [05:12<00:00, ...]"
    m = re.findall(rf"\[{name}\]: 100%.*?\[(\d+:\d+(?::\d+)?)<", log)
    parts = [int(x) for x in m[-1].split(":")]
    return sum(v * 60 ** i for i, v in enumerate(reversed(parts)))

train_s, val_s = phase_seconds("Train"), phase_seconds("Val")
tr, va = train_s / N_TRAIN, val_s / N_VAL
epoch_h = (tr * FULL_TRAIN + va * FULL_VAL) / 3600
print(f"train      : {tr:.2f} s/image  ({train_s} s for {N_TRAIN})")
print(f"validation : {va:.2f} s/image  ({val_s} s for {N_VAL})")
print(f"data load  : {load_s_per_img:.2f} s/image (4 workers)")
print(f"wall time  : {wall / 60:.1f} min for this run")
print(f"full epoch : {epoch_h:.2f} h  ->  10 epochs {10 * epoch_h:.1f} h, 20 epochs {20 * epoch_h:.1f} h, 50 epochs {50 * epoch_h:.1f} h")